# Datamine DECODE Process Example

This notebook demonstrates how to configure and run the **`decode`** process wrapper in `dmstudio`.

## Process Description

## DECODE

Decode a named field of the input file through a dictionary file, writing a new output file containing all the fields of the input file, plus the new decoded field, which is given the name supplied in symbolic field * **TEXT**.

A _dictionary file_ is a database file containing two fields; **CODE** and **TEXT**. Each may be either an alphanumeric or a numeric field. There should be only one value of each **CODE** field; if there are multiple values, then only the first one will be taken. The dictionary file must be sorted on the **CODE** field. If it is not, use the **[SORT](<sortx.md>)** process prior to using the **DECODE** process. The maximum length of the **CODE** field (if alphanumeric) is 40 characters, and the maximum length of the **TEXT** field (if alphanumeric) is 80 characters. The **CODE** field should match in type and length the field to be decoded.

The dictionary **CODE** field is matched against the field specified as symbolic field * **CODE**. If the value matches, then the * **TEXT** field in the output file will contain the value from the **TEXT** dictionary field. If the * **CODE** field is not matched, the * **TEXT** field is set to the * **CODE** field, as closely as field lengths permit. If the fields are different types (i.e. not both numeric or alphanumeric), then the * **TEXT** field is set to absent data (if numeric field) or blank (if alpha field).

Typical uses of the **DECODE** process are to convert geological coding to a text string, or to convert from one language to another. The method is, however, very general, and may be used for example to decode one set of numbers from another; for example, to set slope angles as a function of a numeric rock type.

The dictionary file is first read into your application's virtual array, and is searched using a binary chop search. This is why it must be sorted on the **CODE** field.

**Note** : the default value of the * **TEXT** field in the output file is set to the default value of the **TEXT** field in the dictionary. This can be used to advantage, for example when setting up slope angles in an orebody model from a rocktype code using a dictionary. The default value of the **TEXT** field (representing the SLOPE angle) should be set to the required default **SLOPE** field value.

### Input Files:

* **in** (Undefined):
  File to be decoded. This contains the field **CODE** (real name specified below) which
  is translated through the dictionary file **DICT** matching with the **CODE** field, and
  writing the equivalent **TEXT** field from the dictionary to the output file as field

## **TEXT**.

  Required=Yes

* **dict** (Dictionary):
  Dictionary file (fields **CODE** and **TEXT**). There should be only one occurrence of
  each **CODE** value. This file MUST be sorted on **CODE**.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Decoded file. An exact copy of the input file with the decoded field **TEXT** added.
  Required=Yes

### Fields:

* **code** (Character : IN):
  Field to be translated in input file.
  Default=Undefined
  Required=Yes

* **text** (Character : OUT):
  Field holding translated text, on output file.
  Default=Undefined
  Required=Yes

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('decode')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute decode
print("Running decode...")
dm_cmd.decode(
    in_i='t_assays',  # required
    dict_i='t_input_file',  # required
    out_o='t_decode_out',  # required
    code_f='optional',  # required
    text_f='optional',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("decode execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_decode_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")